# 🧩 Mini-Lab: LLM-as-Judge Evaluation

**Module 9: LLM Evaluation** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why LLM-as-judge is needed when reference-based metrics fall short
2. **Implement** an LLM-as-judge evaluator that scores generated text on subjective quality dimensions
3. **Compare** LLM judge scores across multiple outputs to identify quality differences

## Target Concepts

| Concept | Description |
|---------|-------------|
| LLM-as-Judge | Using an LLM to evaluate the quality of another LLM's output — enabling scalable assessment of subjective dimensions like helpfulness, clarity, and tone |

## Why LLM-as-Judge?

In the previous mini-lab (`mini-reference-eval`), we used BLEU and ROUGE to compare generated text against a known reference. These metrics work well when there's **one correct answer**.

But many LLM tasks have **no single correct answer**:

| Task | Why Reference Metrics Struggle |
|------|-------------------------------|
| "Write a helpful email reply" | Many valid replies exist |
| "Explain quantum computing simply" | Quality depends on clarity, not exact wording |
| "Summarize this article in a friendly tone" | Tone is subjective |

**LLM-as-judge** solves this by using a strong LLM to evaluate outputs on subjective criteria like helpfulness, clarity, and accuracy — the same way a human reviewer would, but at scale.

## Setup

In [4]:
import os
from dotenv import load_dotenv
import openai
from IPython.display import display, Markdown

load_dotenv()

client = openai.OpenAI()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Evaluation Data

We'll evaluate two different LLM-generated answers to the same question. One is detailed and helpful; the other is vague and incomplete.

In [5]:
question = "What are the benefits of using version control like Git?"

response_a = (
    "Git provides several key benefits: it tracks every change to your code so you can "
    "revert mistakes, enables multiple developers to work on the same project without "
    "conflicts through branching, and maintains a complete history of your project. "
    "It also integrates with platforms like GitHub for code review and collaboration."
)

response_b = (
    "Version control is good. It helps with code. You should use it for your projects."
)

md(f"**Question:** {question}\n\n"
   f"**Response A:** {response_a}\n\n"
   f"**Response B:** {response_b}")

**Question:** What are the benefits of using version control like Git?

**Response A:** Git provides several key benefits: it tracks every change to your code so you can revert mistakes, enables multiple developers to work on the same project without conflicts through branching, and maintains a complete history of your project. It also integrates with platforms like GitHub for code review and collaboration.

**Response B:** Version control is good. It helps with code. You should use it for your projects.

## Building the LLM Judge

The core idea is simple: we write a **judge prompt** that tells the LLM what to evaluate and how to score it. The judge returns a numeric score and a brief justification.

We'll evaluate on three dimensions:
- **Helpfulness** — Does it actually answer the question?
- **Specificity** — Does it provide concrete details rather than vague statements?
- **Clarity** — Is it well-organized and easy to understand?

In [6]:
JUDGE_PROMPT = """You are an expert evaluator. Score the following response to a user question.

**Question:** {question}

**Response:** {response}

Evaluate the response on these three criteria. For each, give a score from 1 (worst) to 5 (best) and a one-sentence justification.

Format your answer EXACTLY like this:
Helpfulness: <score>/5 - <justification>
Specificity: <score>/5 - <justification>
Clarity: <score>/5 - <justification>
"""

print("Judge prompt template ready.")

Judge prompt template ready.


In [7]:
def llm_judge(question, response, model="gpt-4o-mini"):
    """Use an LLM to judge the quality of a response."""
    prompt = JUDGE_PROMPT.format(question=question, response=response)
    
    result = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0  # Deterministic for consistent evaluation
    )
    return result.choices[0].message.content

## Judging Response A

In [9]:
judgment_a = llm_judge(question, response_a)
md(f"### Response A Evaluation\n\n{judgment_a}")

### Response A Evaluation

Helpfulness: 5/5 - The response effectively outlines the main benefits of using Git, addressing key aspects that are valuable to users considering version control.  
Specificity: 4/5 - While the response covers important features, it could provide more specific examples or details about how these benefits are realized in practice.  
Clarity: 5/5 - The response is clear and easy to understand, using straightforward language that conveys the information effectively.

## Judging Response B

In [10]:
judgment_b = llm_judge(question, response_b)
md(f"### Response B Evaluation\n\n{judgment_b}")

### Response B Evaluation

Helpfulness: 2/5 - The response provides a very basic acknowledgment of version control's benefits but lacks detailed information on how it specifically helps with code and projects.  
Specificity: 1/5 - The response is vague and does not mention any specific benefits or features of version control systems like Git.  
Clarity: 3/5 - While the response is clear in its language, it is overly simplistic and does not convey a comprehensive understanding of the topic.

## Parsing Judge Scores

To use judge scores programmatically (e.g., in an evaluation pipeline), we need to parse the numeric scores from the judge's text output.

In [11]:
import re

def parse_scores(judgment_text):
    """Extract numeric scores from judge output."""
    scores = {}
    for criterion in ["Helpfulness", "Specificity", "Clarity"]:
        match = re.search(rf"{criterion}:\s*(\d)/5", judgment_text)
        if match:
            scores[criterion] = int(match.group(1))
    return scores

scores_a = parse_scores(judgment_a)
scores_b = parse_scores(judgment_b)

print(f"Response A scores: {scores_a}")
print(f"Response B scores: {scores_b}")

Response A scores: {'Helpfulness': 5, 'Specificity': 4, 'Clarity': 5}
Response B scores: {'Helpfulness': 2, 'Specificity': 1, 'Clarity': 3}


## Side-by-Side Comparison

In [12]:
lines = [
    "| Criterion | Response A | Response B |",
    "|-----------|-----------|-----------|" 
]

for criterion in ["Helpfulness", "Specificity", "Clarity"]:
    sa = scores_a.get(criterion, "?")
    sb = scores_b.get(criterion, "?")
    lines.append(f"| {criterion} | {sa}/5 | {sb}/5 |")

avg_a = sum(scores_a.values()) / len(scores_a) if scores_a else 0
avg_b = sum(scores_b.values()) / len(scores_b) if scores_b else 0
lines.append(f"| **Average** | **{avg_a:.1f}/5** | **{avg_b:.1f}/5** |")

md("\n".join(lines))

| Criterion | Response A | Response B |
|-----------|-----------|-----------|
| Helpfulness | 5/5 | 2/5 |
| Specificity | 4/5 | 1/5 |
| Clarity | 5/5 | 3/5 |
| **Average** | **4.7/5** | **2.0/5** |

## Pairwise Comparison: Head-to-Head Judging

Another common LLM-as-judge pattern is **pairwise comparison** — showing the judge both responses at once and asking it to pick the better one. This avoids absolute scoring biases.

In [13]:
PAIRWISE_PROMPT = """You are an expert evaluator. Given a question and two responses, decide which response is better.

**Question:** {question}

**Response A:** {response_a}

**Response B:** {response_b}

Which response is better overall? Consider helpfulness, specificity, and clarity.

Format your answer EXACTLY like this:
Winner: <A or B>
Reason: <one-sentence explanation>
"""

result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PAIRWISE_PROMPT.format(
        question=question, response_a=response_a, response_b=response_b
    )}],
    temperature=0
)

md(f"### Pairwise Judgment\n\n{result.choices[0].message.content}")

### Pairwise Judgment

Winner: A  
Reason: Response A provides specific benefits of using Git, including tracking changes, enabling collaboration, and integration with platforms, making it more informative and helpful than Response B.

## Key Design Decisions for LLM-as-Judge

When building your own LLM judge, keep these in mind:

| Decision | Recommendation |
|----------|---------------|
| **Temperature** | Use `0` for consistent, reproducible scores |
| **Output format** | Request a structured format so scores are easy to parse |
| **Judge model** | Use the strongest model available — a weak judge gives unreliable scores |
| **Scoring mode** | Use pointwise (1–5) for absolute quality; pairwise (A vs B) for ranking |
| **Position bias** | In pairwise mode, swap A/B order and check if the winner changes |

## 🎯 Summary

### Key Takeaways

1. **LLM-as-judge fills the gap** — when there's no single correct answer, reference-based metrics like BLEU and ROUGE can't assess subjective quality dimensions like helpfulness and clarity
2. **Pointwise scoring** — ask the judge to rate a single response on specific criteria (e.g., 1–5 scale) for absolute quality measurement
3. **Pairwise comparison** — show the judge two responses side-by-side to determine which is better, avoiding absolute scoring biases
4. **Structured output and low temperature** — requesting a consistent format and using temperature 0 makes judge scores parseable and reproducible